# Cogongrass Close-Up Classifier (Colab) — v4: stratified test split

2000 images/class, near-duplicate dedup, **stratified 70/15/15** train/val/test. Early stopping uses **val**; the **test** set is held out and scored exactly once. ResNet50, two-phase fine-tuning, Grad-CAM verification.

**Runtime:** `Runtime > Change runtime type > GPU (T4)`.

In [ ]:
# 1. GPU check
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print('GPU:', torch.cuda.get_device_name(0))
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## 2. Config

In [ ]:
CLASSES = {
    'cogongrass': ['Imperata cylindrica'],
    'other_grass': [
        'Imperata brasiliensis', 'Saccharum spontaneum', 'Cenchrus purpureus',
        'Setaria', 'Andropogon virginicus', 'Pennisetum', 'Miscanthus sinensis',
        'Sorghum halepense', 'Cortaderia selloana', 'Phragmites australis',
    ],
}

IMAGES_PER_CLASS = 2000
VAL_FRACTION  = 0.15
TEST_FRACTION = 0.15        # held out, scored once
DEDUP_BITS    = 4           # aHash hamming distance <= this -> treated as duplicate
IMG_SIZE      = 224
PHOTO_SIZE    = 'medium'
DATA_DIR      = '/content/data'
SEED          = 42

BATCH_SIZE    = 32
WARMUP_EPOCHS = 4
MAX_EPOCHS    = 80
PATIENCE      = 10
HEAD_LR       = 1e-3
BACKBONE_LR   = 1e-4
WEIGHT_DECAY  = 1e-4
LABEL_SMOOTH  = 0.05

## 3. Download labeled images from iNaturalist (one photo per observation)

In [ ]:
import os, time, requests, random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

random.seed(SEED)
API = 'https://api.inaturalist.org/v1'
HEADERS = {'User-Agent': 'cogongrass-baseline/1.0'}

def resolve_taxon(name):
    r = requests.get(f'{API}/taxa', params={'q': name, 'per_page': 1}, headers=HEADERS, timeout=30)
    r.raise_for_status(); res = r.json()['results']
    if not res: print(f'  !! no taxon for "{name}"'); return None
    t = res[0]; print(f'  {name} -> id {t["id"]} ({t["name"]})'); return t['id']

def obs_photos_for_taxon(taxon_id, n):
    """Return up to n (observation_id, photo_url) pairs, one photo per observation."""
    out, page = [], 1
    while len(out) < n and page <= 60:
        r = requests.get(f'{API}/observations', headers=HEADERS, timeout=30, params={
            'taxon_id': taxon_id, 'photos': 'true', 'quality_grade': 'research',
            'per_page': 200, 'page': page, 'order_by': 'votes'})
        r.raise_for_status(); results = r.json()['results']
        if not results: break
        for obs in results:
            if obs.get('photos'):
                out.append((obs['id'], obs['photos'][0]['url'].replace('square', PHOTO_SIZE)))
        page += 1; time.sleep(0.7)
    return out[:n]

def download(args):
    url, path = args
    if path.exists(): return True
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        if r.status_code == 200 and r.content: path.write_bytes(r.content); return True
    except Exception: pass
    return False

for cls, species in CLASSES.items():
    out = Path(DATA_DIR) / cls; out.mkdir(parents=True, exist_ok=True)
    per_species = max(1, IMAGES_PER_CLASS // len(species))
    print(f'\n== {cls} (target {IMAGES_PER_CLASS}, {per_species}/species) ==')
    jobs = []
    for sp in species:
        tid = resolve_taxon(sp)
        if tid is None: continue
        for obs_id, url in obs_photos_for_taxon(tid, per_species):
            jobs.append((url, out / f'{tid}_{obs_id}.jpg'))  # obs id in filename = traceable
    with ThreadPoolExecutor(max_workers=16) as ex:
        ok = sum(ex.map(download, jobs))
    print(f'  downloaded {ok}/{len(jobs)} into {out}')

In [ ]:
# 3b. Drop corrupt files, then drop near-duplicate images (perceptual aHash)
import numpy as np
from PIL import Image
from pathlib import Path

bad = 0
for p in Path(DATA_DIR).rglob('*.jpg'):
    try: Image.open(p).convert('RGB').verify()
    except Exception: p.unlink(); bad += 1
print(f'removed {bad} corrupt files')

def ahash(path):
    a = np.asarray(Image.open(path).convert('L').resize((8, 8)), dtype=np.float32)
    return np.packbits((a > a.mean()).flatten().astype(np.uint8)).view(np.uint64)[0]

def popcount(x):  # bits set, per element of a uint64 array
    return np.unpackbits(x.view(np.uint8).reshape(-1, 8), axis=1).sum(1)

# dedup within each class folder so we don't accidentally erase a real cross-class look-alike
for cls in CLASSES:
    paths = sorted((Path(DATA_DIR) / cls).glob('*.jpg'))
    hashes = np.array([ahash(p) for p in paths], dtype=np.uint64)
    kept_h, removed = [], 0
    for p, h in zip(paths, hashes):
        if kept_h:
            d = popcount(np.bitwise_xor(np.array(kept_h, dtype=np.uint64), h))
            if (d <= DEDUP_BITS).any(): p.unlink(); removed += 1; continue
        kept_h.append(h)
    print(f'{cls}: kept {len(kept_h)}, removed {removed} near-duplicates')

## 4. Datasets, transforms, and the stratified 70/15/15 split

`train_test_split(..., stratify=labels)` forces every split to hold the same class proportions. Test is carved out and never touched until the final cell.

In [ ]:
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
from collections import Counter

torch.manual_seed(SEED)
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomRotation(20), transforms.ColorJitter(0.25, 0.25, 0.25, 0.05),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD), transforms.RandomErasing(p=0.25),
])
eval_tf = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.15)), transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])

base_train = datasets.ImageFolder(DATA_DIR, transform=train_tf)
base_eval  = datasets.ImageFolder(DATA_DIR, transform=eval_tf)
class_names = base_train.classes
print('classes:', class_names)

# Stratified 70/15/15: peel off the held-out 30%, then halve it into val/test.
N = len(base_train)
labels = list(base_train.targets)
rel_test = TEST_FRACTION / (VAL_FRACTION + TEST_FRACTION)
train_idx, temp_idx = train_test_split(range(N), test_size=VAL_FRACTION + TEST_FRACTION,
                                       stratify=labels, random_state=SEED)
val_idx, test_idx = train_test_split(temp_idx, test_size=rel_test,
                                     stratify=[labels[i] for i in temp_idx], random_state=SEED)

train_set = Subset(base_train, train_idx)   # augmented
val_set   = Subset(base_eval,  val_idx)     # clean
test_set  = Subset(base_eval,  test_idx)    # clean, held out

def balance(idx):
    c = Counter(labels[i] for i in idx)
    return {class_names[k]: c[k] for k in sorted(c)}
print('balance -> train', balance(train_idx), '| val', balance(val_idx), '| test', balance(test_idx))

train_loader = DataLoader(train_set, BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'train {len(train_set)}  val {len(val_set)}  test {len(test_set)}')

## 5. Model — ResNet50

In [ ]:
import torch.nn as nn
from torchvision import models

model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.fc = nn.Linear(model.fc.in_features, len(class_names)); model = model.to(device)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)
scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))

def set_backbone_trainable(flag):
    for n_, p in model.named_parameters():
        if not n_.startswith('fc.'): p.requires_grad = flag

def run_epoch(loader, optimizer=None):
    train = optimizer is not None
    model.train() if train else model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
                out = model(x); loss = criterion(out, y)
            if train:
                optimizer.zero_grad(); scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update()
            loss_sum += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item(); total += x.size(0)
    return loss_sum / total, correct / total

## 6. Phase 1 — warm up the head

In [ ]:
set_backbone_trainable(False)
opt = torch.optim.Adam(model.fc.parameters(), lr=HEAD_LR, weight_decay=WEIGHT_DECAY)
for epoch in range(1, WARMUP_EPOCHS + 1):
    tl, ta = run_epoch(train_loader, opt); vl, va = run_epoch(val_loader)
    print(f'warmup {epoch:2d}  train_loss {tl:.3f} acc {ta:.3f} | val_loss {vl:.3f} acc {va:.3f}')

## 7. Phase 2 — fine-tune everything, early stopping on val

In [ ]:
import copy
set_backbone_trainable(True)
bb = [p for n_, p in model.named_parameters() if not n_.startswith('fc.')]
hd = [p for n_, p in model.named_parameters() if n_.startswith('fc.')]
opt = torch.optim.AdamW([{'params': bb, 'lr': BACKBONE_LR}, {'params': hd, 'lr': HEAD_LR}],
                        weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.3, patience=4)

best_vl = float('inf'); best_state = copy.deepcopy(model.state_dict())
best_epoch, since = 0, 0; history = []
for epoch in range(1, MAX_EPOCHS + 1):
    tl, ta = run_epoch(train_loader, opt); vl, va = run_epoch(val_loader)
    sched.step(vl); history.append((tl, ta, vl, va)); flag = ''
    if vl < best_vl - 1e-4:
        best_vl, best_epoch, since = vl, epoch, 0
        best_state = copy.deepcopy(model.state_dict()); flag = '  <- best'
    else: since += 1
    print(f'epoch {epoch:2d}  train_loss {tl:.3f} acc {ta:.3f} | val_loss {vl:.3f} acc {va:.3f} | lr {opt.param_groups[0]["lr"]:.1e}{flag}')
    if since >= PATIENCE: print(f'\nearly stop after {epoch} epochs'); break

model.load_state_dict(best_state)
torch.save(model.state_dict(), '/content/cogongrass_best.pt')
print(f'restored best from epoch {best_epoch} (val_loss {best_vl:.3f}, val_acc {history[best_epoch-1][3]:.3f})')

In [ ]:
import matplotlib.pyplot as plt
tl, ta, vl, va = zip(*history); ep = range(1, len(history) + 1)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(ep, tl, label='train'); ax[0].plot(ep, vl, label='val'); ax[0].set_title('loss'); ax[0].legend()
ax[1].plot(ep, ta, label='train'); ax[1].plot(ep, va, label='val'); ax[1].set_title('accuracy'); ax[1].legend()
for a in ax: a.axvline(best_epoch, ls='--', c='gray'); a.set_xlabel('epoch')
plt.show()

## 8. Final scores — val (for reference) and **test** (the real number)

Test is evaluated here for the first time.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

def evaluate(loader):
    model.eval(); yt, yp = [], []
    with torch.no_grad():
        for x, y in loader:
            yp += model(x.to(device)).argmax(1).cpu().tolist(); yt += y.tolist()
    return yt, yp

for split_name, loader in [('VALIDATION', val_loader), ('TEST', test_loader)]:
    yt, yp = evaluate(loader)
    print(f'\n===== {split_name} =====')
    print(classification_report(yt, yp, target_names=class_names, digits=3))
    print('confusion (rows=true, cols=pred):')
    print('         ' + '  '.join(f'{c[:8]:>8}' for c in class_names))
    for name, row in zip(class_names, confusion_matrix(yt, yp)):
        print(f'{name[:8]:>8} ' + '  '.join(f'{v:>8}' for v in row))

## 9. Grad-CAM on the held-out test set

In [ ]:
import numpy as np, matplotlib.pyplot as plt
acts, grads = {}, {}
tl_layer = model.layer4[-1]
h1 = tl_layer.register_forward_hook(lambda m, i, o: acts.__setitem__('v', o.detach()))
h2 = tl_layer.register_full_backward_hook(lambda m, gi, go: grads.__setitem__('v', go[0].detach()))
inv = transforms.Normalize([-m/s for m, s in zip(MEAN, STD)], [1/s for s in STD])

def gradcam(x):
    model.eval(); x = x.unsqueeze(0).to(device); out = model(x); cls = out.argmax(1)
    model.zero_grad(); out[0, cls].backward()
    w = grads['v'].mean(dim=(2, 3), keepdim=True)
    cam = (w * acts['v']).sum(1).clamp(min=0)[0]
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    cam = torch.nn.functional.interpolate(cam[None, None], (IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)[0, 0]
    return cam.cpu().numpy(), cls.item(), out.softmax(1)[0, cls].item()

x, y = next(iter(test_loader))
plt.figure(figsize=(15, 9))
for i in range(min(12, len(x))):
    cam, pred, conf = gradcam(x[i])
    plt.subplot(3, 4, i + 1)
    plt.imshow(inv(x[i]).permute(1, 2, 0).clamp(0, 1).numpy()); plt.imshow(cam, cmap='jet', alpha=0.45)
    ok = pred == y[i].item()
    plt.title(f'{class_names[pred]} {conf:.2f} (true {class_names[y[i]]})', color='green' if ok else 'red', fontsize=9)
    plt.axis('off')
plt.tight_layout(); plt.show(); h1.remove(); h2.remove()

In [ ]:
# 10. Save / download
from google.colab import files
files.download('/content/cogongrass_best.pt')